In [ ]:
!pip install -q torch torchvision

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.datasets import STL10
from torch.utils.data import DataLoader, ConcatDataset
import numpy as np
import os

SAVE_DIR = "/content/drive/MyDrive/simclr_stl10"
os.makedirs(SAVE_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
class TwoCropTransform:
    """
    Takes a single image and returns two independently
    augmented versions of it as a tuple (view_a, view_b).
    """
    def __init__(self, transform):
        self.transform = transform

    def __call__(self, x):
        return self.transform(x), self.transform(x)

# SimCLR augmentation policy from Chen et al. (2020)
# Colour jitter + random crop are the most important
simclr_transform = transforms.Compose([
    transforms.RandomResizedCrop(size=96),           # random crop, resize to 96x96
    transforms.RandomHorizontalFlip(p=0.5),          # 50% chance of horizontal flip
    transforms.RandomApply([
        transforms.ColorJitter(
            brightness=0.4,
            contrast=0.4,
            saturation=0.4,
            hue=0.1
        )
    ], p=0.8),                                       # 80% chance of colour jitter
    transforms.RandomGrayscale(p=0.2),               # 20% chance of greyscale
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [ ]:
print("Loading STL-10 datasets...")

# Unlabelled split: 100,000 images, no labels
unlabelled = STL10(
    root='/content/data',
    split='unlabeled',
    download=True,
    transform=TwoCropTransform(simclr_transform)
)

# Labelled train split: 5,000 images, labels ignored during SimCLR
labelled = STL10(
    root='/content/data',
    split='train',
    download=True,
    transform=TwoCropTransform(simclr_transform)
)

# Combine both splits
full_dataset = ConcatDataset([unlabelled, labelled])

loader = DataLoader(
    full_dataset,
    batch_size=256,          # larger batch = more negatives = better contrastive learning
    shuffle=True,
    num_workers=4,
    pin_memory=True,         # faster GPU transfer
    drop_last=True           # NT-Xent requires consistent batch size
)

print(f"Total images: {len(full_dataset)}")
print(f"Batches per epoch: {len(loader)}")

In [ ]:
class SimCLR(nn.Module):
    def __init__(self, projection_dim=128):
        super().__init__()

        # Backbone: ResNet50 with classification head removed
        # This time weights are NOT frozen — they update during training
        resnet = models.resnet50(weights=None)  # random init, we train from scratch
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])  # remove fc layer

        # Projection head: 2-layer MLP
        # Maps 2048-dim backbone output to projection_dim
        # This is used ONLY during training and discarded afterwards
        self.projection_head = nn.Sequential(
            nn.Linear(2048, 2048),
            nn.ReLU(),
            nn.Linear(2048, projection_dim)
        )

    def forward(self, x):
        # Extract features from backbone
        h = self.backbone(x)
        h = h.view(h.size(0), -1)          # flatten: (batch, 2048, 1, 1) -> (batch, 2048)

        # Project to contrastive space
        z = self.projection_head(h)
        return h, z                          # return both: h for embeddings, z for loss


model = SimCLR(projection_dim=128).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
class NTXentLoss(nn.Module):
    """
    Normalised Temperature-scaled Cross Entropy Loss.
    For each view, its positive pair is its other augmented version.
    Everything else in the batch is a negative.

    temperature: controls how sharply the distribution is peaked.
    Lower temperature = harder negatives = stronger learning signal.
    """
    def __init__(self, temperature=0.5):
        super().__init__()
        self.temperature = temperature

    def forward(self, z_a, z_b):
        batch_size = z_a.size(0)

        # L2 normalise both sets of projections
        z_a = F.normalize(z_a, p=2, dim=1)
        z_b = F.normalize(z_b, p=2, dim=1)

        # Concatenate: [z_a_1, z_a_2, ..., z_b_1, z_b_2, ...]
        z = torch.cat([z_a, z_b], dim=0)   # shape: (2*batch_size, projection_dim)

        # Compute all pairwise cosine similarities
        sim = torch.mm(z, z.T) / self.temperature  # shape: (2N, 2N)

        # Mask out self-similarity (diagonal)
        mask = torch.eye(2 * batch_size, dtype=torch.bool).to(device)
        sim.masked_fill_(mask, float('-inf'))

        # Positive pairs: (i, i+N) and (i+N, i)
        # For view i in z_a, its positive is view i in z_b (index i+N)
        labels = torch.cat([
            torch.arange(batch_size, 2 * batch_size),
            torch.arange(batch_size)
        ]).to(device)

        loss = F.cross_entropy(sim, labels)
        return loss


criterion = NTXentLoss(temperature=0.5)

# ── CELL 8: Optimiser and scheduler ───────────────────────
# LARS optimiser is ideal for SimCLR but Adam works fine for Colab
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

# Cosine annealing: gradually reduces learning rate over training
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=100,      # number of epochs
    eta_min=1e-6    # minimum learning rate
)

In [ ]:
EPOCHS = 100
best_loss = float('inf')

print("Starting SimCLR training...")
print(f"Epochs: {EPOCHS}, Batch size: 256, Temperature: 0.5")
print("-" * 50)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for batch_idx, (views, _) in enumerate(loader):
        # views is a tuple (view_a, view_b) from TwoCropTransform
        view_a, view_b = views
        view_a = view_a.to(device)
        view_b = view_b.to(device)

        optimizer.zero_grad()

        # Forward pass: get projections for both views
        _, z_a = model(view_a)
        _, z_b = model(view_b)

        # Compute contrastive loss
        loss = criterion(z_a, z_b)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    scheduler.step()

    avg_loss = total_loss / len(loader)
    lr = scheduler.get_last_lr()[0]

    print(f"Epoch [{epoch+1:3d}/{EPOCHS}] Loss: {avg_loss:.4f} | LR: {lr:.6f}")

    # Save checkpoint every 10 epochs and whenever loss improves
    if (epoch + 1) % 10 == 0 or avg_loss < best_loss:
        checkpoint = {
            'epoch': epoch + 1,
            'backbone_state_dict': model.backbone.state_dict(),
            'full_model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_loss
        }
        path = f"{SAVE_DIR}/simclr_epoch_{epoch+1}.pt"
        torch.save(checkpoint, path)
        print(f"  Checkpoint saved: {path}")

        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(checkpoint, f"{SAVE_DIR}/simclr_best.pt")
            print(f"  New best model saved.")

print("Training complete.")

In [ ]:
torch.save(
    model.backbone.state_dict(),
    f"{SAVE_DIR}/simclr_backbone_final.pt"
)
print(f"Final backbone saved to {SAVE_DIR}/simclr_backbone_final.pt")
print("Download this file and place it in your ./models/ directory.")

In [ ]:
model.eval()
with torch.no_grad():
    dummy = torch.randn(4, 3, 96, 96).to(device)
    h, z = model(dummy)
    print(f"Backbone output shape: {h.shape}")    # should be (4, 2048)
    print(f"Projection output shape: {z.shape}")  # should be (4, 128)
    print("Sanity check passed.")